# 05 베이스라인 비교 및 메타데이터 효과 검증

04번에서 학습한 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)이 실제로 좋은 모델인지 확인하기 위해 베이스라인과 비교한다.

목적은 다음과 같다.

1. `TF-IDF + Random Forest` 베이스라인을 학습한다.
2. `KcBERT PCA only + MLP`, `Metadata only + MLP` 비교 모델을 학습한다.
3. 04번의 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) 결과와 같은 지표로 성능을 비교한다.
4. 메타데이터 결합이 F1, PR-AUC, Recall 개선에 실제로 기여했는지 확인한다.
5. 06번 최종 모델 선택 및 오답 분석에 필요한 결과를 저장한다.

05번은 아직 별점 정제를 수행하지 않는다. 별점 정제는 06번에서 최종 모델을 선택한 뒤 07번에서 진행한다.

## 1. 라이브러리 로드

- 04번과 같은 평가 지표를 사용한다.
- MLP 계열 비교 모델은 train 데이터에만 SMOTE를 적용한다.
- TF-IDF + Random Forest는 sparse text feature를 사용하므로 SMOTE를 적용하지 않고 `class_weight='balanced'`로 불균형을 보정한다.


In [2]:
import json
import os

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

os.makedirs('outputs', exist_ok=True)

## 2. 데이터 로드 및 split 재현

- `reviews_embeddings_extract.csv`에서 raw KcBERT 임베딩과 raw 메타데이터를 불러온다.
- TF-IDF 베이스라인은 `cleaned_review_text`를 사용한다.
- 04번과 같은 `random_state=42`, `stratify=label` 조건으로 train / validation / test split을 재현한다.

In [ ]:
SEED = 42
LABEL_COL = 'label'
TEXT_COL = 'cleaned_review_text'
MIN_TUNED_THRESHOLD = 0.5

raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')

emb_cols = [f'kcbert_{i}' for i in range(768)]
meta_cols = ['text_length', 'emoji_count', 'photo_count']
hybrid_cols = emb_cols + meta_cols

X_all = raw_df[hybrid_cols].copy()
y_all = raw_df[LABEL_COL].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X_all,
    y_all,
    test_size=0.3,
    random_state=SEED,
    stratify=y_all,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp,
)

train_idx = X_train.index
val_idx = X_val.index
test_idx = X_test.index

text_train = raw_df.loc[train_idx, TEXT_COL].fillna('').astype(str)
text_val = raw_df.loc[val_idx, TEXT_COL].fillna('').astype(str)
text_test = raw_df.loc[test_idx, TEXT_COL].fillna('').astype(str)

print('Raw embedding data:', raw_df.shape)
print('Hybrid train:', X_train.shape)
print('Hybrid validation:', X_val.shape)
print('Hybrid test:', X_test.shape)
print('KcBERT feature 수:', len(emb_cols))
print('metadata feature:', meta_cols)
print('excluded feature:', ['rating'])
print('train label 분포:', y_train.value_counts().sort_index().to_dict())
print('validation label 분포:', y_val.value_counts().sort_index().to_dict())
print('test label 분포:', y_test.value_counts().sort_index().to_dict())

Raw embedding data: (8841, 783)
Hybrid train: (6188, 771)
Hybrid validation: (1326, 771)
Hybrid test: (1327, 771)
KcBERT feature ?: 768
metadata feature: ['text_length', 'emoji_count', 'photo_count']
excluded feature: ['rating']
train label ??: {0: 3983, 1: 2205}
validation label ??: {0: 854, 1: 472}
test label ??: {0: 854, 1: 473}


C:\Users\ICT\AppData\Local\Temp\ipykernel_21532\3283360203.py:6: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv('csv/reviews_embeddings_extract.csv')


## 3. 공통 평가 함수 정의

- 기본 threshold는 `0.5`이다.
- validation 데이터에서 F1이 가장 높은 threshold를 추가로 찾으며, test 데이터는 threshold 선택에 사용하지 않는다.
- 단, threshold가 지나치게 낮아져 일반 리뷰를 과도하게 이벤트 리뷰로 오탐하지 않도록 `0.5 이상` 후보만 사용한다.

In [4]:
def tune_threshold_from_validation(y_true, prob, min_threshold=0.5):
    precisions, recalls, thresholds = precision_recall_curve(y_true, prob)

    if len(thresholds) == 0:
        return float(min_threshold), pd.DataFrame()

    f1s = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-8)
    candidates = pd.DataFrame({
        'threshold': thresholds,
        'precision': precisions[:-1],
        'recall': recalls[:-1],
        'f1': f1s,
    })
    candidates = candidates[candidates['threshold'] >= min_threshold]

    if candidates.empty:
        return float(min_threshold), candidates

    best_threshold = float(candidates.sort_values('f1', ascending=False).iloc[0]['threshold'])
    return best_threshold, candidates.sort_values('f1', ascending=False).reset_index(drop=True)


def metric_row(model_name, feature_set, split, y_true, prob, threshold):
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred, labels=[0, 1])
    return {
        'model': model_name,
        'feature_set': feature_set,
        'split': split,
        'threshold': float(threshold),
        'f1': float(f1_score(y_true, pred)),
        'pr_auc': float(average_precision_score(y_true, prob)),
        'precision': float(precision_score(y_true, pred, zero_division=0)),
        'recall': float(recall_score(y_true, pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, prob)),
        'tn': int(cm[0, 0]),
        'fp': int(cm[0, 1]),
        'fn': int(cm[1, 0]),
        'tp': int(cm[1, 1]),
    }


def evaluate_probabilities(model_name, feature_set, y_val, val_prob, y_test, test_prob):
    best_threshold, threshold_candidates = tune_threshold_from_validation(
        y_val,
        val_prob,
        min_threshold=MIN_TUNED_THRESHOLD,
    )

    rows = [
        metric_row(model_name, feature_set, 'validation_default_0_5', y_val, val_prob, 0.5),
        metric_row(model_name, feature_set, 'validation_tuned_min_0_5', y_val, val_prob, best_threshold),
        metric_row(model_name, feature_set, 'test_default_0_5', y_test, test_prob, 0.5),
        metric_row(model_name, feature_set, 'test_tuned_min_0_5', y_test, test_prob, best_threshold),
    ]
    return rows, best_threshold, threshold_candidates


## 4. 제안 모델 결과 로드

04번에서 이미 학습한 `KcBERT PCA + Metadata Hybrid + MLP` 결과를 불러온다.

이 결과가 05번 비교표에서 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata) 역할을 한다.

In [5]:
proposed_metrics = pd.read_csv('outputs/proposed_mlp_metrics.csv')
proposed_metrics.insert(0, 'feature_set', 'kcbert_pca+metadata')
proposed_metrics.insert(0, 'model', 'proposed_hybrid_mlp_04')

with open('outputs/proposed_mlp_selected_config.json', 'r', encoding='utf-8') as f:
    proposed_config = json.load(f)

proposed_metrics


,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,proposed_hybrid_mlp_04,kcbert_pca+metadata,validation_default_0_5,0.500000,0.444012,0.406997,0.410811,0.483051,0.556058,527,327,244,228
1,proposed_hybrid_mlp_04,kcbert_pca+metadata,validation_tuned_min_0_5,0.503774,0.444444,0.406997,0.411552,0.483051,0.556058,528,326,244,228
2,proposed_hybrid_mlp_04,kcbert_pca+metadata,test_default_0_5,0.500000,0.427403,0.442284,0.413861,0.441860,0.578432,558,296,264,209
3,proposed_hybrid_mlp_04,kcbert_pca+metadata,test_tuned_min_0_5,0.503774,0.427105,0.442284,0.415170,0.439746,0.578432,561,293,265,208


## 5. TF-IDF + Random Forest 베이스라인

텍스트만 사용하는 전통적 베이스라인이다.

- 입력: `cleaned_review_text`
- feature: TF-IDF unigram/bigram
- classifier: RandomForestClassifier
- 불균형 보정: `class_weight='balanced`

이 모델과 제안 모델 (Hybrid MLP: KcBERT PCA + Metadata)을 비교하면, KcBERT 임베딩과 메타데이터 결합 전략이 단순 텍스트 베이스라인보다 나은지 확인할 수 있다.

In [6]:
tfidf_rf = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=20000,
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
    )),
    ('rf', RandomForestClassifier(
        n_estimators=300,
        random_state=SEED,
        n_jobs=-1,
        class_weight='balanced',
    )),
])

tfidf_rf.fit(text_train, y_train)

tfidf_val_prob = tfidf_rf.predict_proba(text_val)[:, 1]
tfidf_test_prob = tfidf_rf.predict_proba(text_test)[:, 1]

tfidf_rows, tfidf_best_threshold, tfidf_threshold_candidates = evaluate_probabilities(
    'baseline_tfidf_random_forest',
    'cleaned_text_tfidf',
    y_val,
    tfidf_val_prob,
    y_test,
    tfidf_test_prob,
)

joblib.dump({
    'model': tfidf_rf,
    'model_name': 'baseline_tfidf_random_forest',
    'text_col': TEXT_COL,
    'target_col': LABEL_COL,
    'default_threshold': 0.5,
    'best_threshold': tfidf_best_threshold,
}, 'outputs/baseline_tfidf_random_forest_model.joblib')

print('TF-IDF + Random Forest best threshold:', tfidf_best_threshold)
display(pd.DataFrame(tfidf_rows).round(4))


TF-IDF + Random Forest best threshold: 0.5015818737837087


,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,baseline_tfidf_random_forest,cleaned_text_tfidf,validation_default_0_5,0.5000,0.2609,0.4015,0.4462,0.1843,0.5482,746,108,385,87
1,baseline_tfidf_random_forest,cleaned_text_tfidf,validation_tuned_min_0_5,0.5016,0.2613,0.4015,0.4485,0.1843,0.5482,747,107,385,87
2,baseline_tfidf_random_forest,cleaned_text_tfidf,test_default_0_5,0.5000,0.2918,0.4381,0.5189,0.2030,0.5837,765,89,377,96
3,baseline_tfidf_random_forest,cleaned_text_tfidf,test_tuned_min_0_5,0.5016,0.2927,0.4381,0.5246,0.2030,0.5837,767,87,377,96


## 6. KcBERT PCA only / Metadata only + MLP 비교

 제안 모델이 좋은 이유가 텍스트 임베딩 때문인지, 메타데이터 때문인지, 둘의 결합 때문인지 확인하기 위한 Ablation Study를 설계한다.

- `KcBERT PCA only + MLP`: KcBERT 임베딩 PCA feature만 사용
- `Metadata only + MLP`: metadata feature만 사용
- `KcBERT PCA + Metadata Hybrid + MLP`: 04번 결과 사용

MLP 구조는 04번에서 선택된 mlp_32 설정을 그대로 사용한다.

In [7]:
def make_mlp_model(cols, use_emb, use_meta, random_state=SEED):
    transformers = []
    if use_emb:
        transformers.append(('kcbert_pca', PCA(n_components=0.90, random_state=random_state), emb_cols))
    if use_meta:
        transformers.append(('metadata_scaler', StandardScaler(), meta_cols))

    preprocessor = ColumnTransformer(transformers=transformers, remainder='drop')

    return ImbPipeline([
        ('preprocess', preprocessor),
        ('smote', SMOTE(random_state=random_state)),
        ('mlp', MLPClassifier(
            hidden_layer_sizes=tuple(proposed_config['hidden_layer_sizes']),
            activation='relu',
            alpha=proposed_config['alpha'],
            batch_size=64,
            learning_rate_init=1e-3,
            max_iter=200,
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            random_state=random_state,
        )),
    ])


def fit_mlp_variant(model_name, feature_set, cols, use_emb, use_meta):
    X_train_variant = X_train[cols]
    X_val_variant = X_val[cols]
    X_test_variant = X_test[cols]

    model = make_mlp_model(cols, use_emb=use_emb, use_meta=use_meta, random_state=SEED)
    model.fit(X_train_variant, y_train)

    val_prob = model.predict_proba(X_val_variant)[:, 1]
    test_prob = model.predict_proba(X_test_variant)[:, 1]

    rows, best_threshold, threshold_candidates = evaluate_probabilities(
        model_name,
        feature_set,
        y_val,
        val_prob,
        y_test,
        test_prob,
    )
    return model, rows, best_threshold, threshold_candidates


pca_only_model, pca_only_rows, pca_only_threshold, pca_only_candidates = fit_mlp_variant(
    'ablation_mlp_kcbert_pca_only',
    'kcbert_pca_only',
    emb_cols,
    use_emb=True,
    use_meta=False,
)

metadata_only_model, metadata_only_rows, metadata_only_threshold, metadata_only_candidates = fit_mlp_variant(
    'ablation_mlp_metadata_only',
    'metadata_only',
    meta_cols,
    use_emb=False,
    use_meta=True,
)

joblib.dump({
    'model': pca_only_model,
    'model_name': 'ablation_mlp_kcbert_pca_only',
    'feature_cols': emb_cols,
    'emb_cols': emb_cols,
    'meta_cols': [],
    'target_col': LABEL_COL,
    'input_type': 'tabular_raw',
    'default_threshold': 0.5,
    'best_threshold': pca_only_threshold,
}, 'outputs/ablation_mlp_kcbert_pca_only_model.joblib')

joblib.dump({
    'model': metadata_only_model,
    'model_name': 'ablation_mlp_metadata_only',
    'feature_cols': meta_cols,
    'emb_cols': [],
    'meta_cols': meta_cols,
    'target_col': LABEL_COL,
    'input_type': 'tabular_raw',
    'default_threshold': 0.5,
    'best_threshold': metadata_only_threshold,
}, 'outputs/ablation_mlp_metadata_only_model.joblib')

print('KcBERT PCA only best threshold:', pca_only_threshold)
print('Metadata only best threshold:', metadata_only_threshold)
display(pd.DataFrame(pca_only_rows + metadata_only_rows).round(4))

KcBERT PCA only best threshold: 0.5067211727217427
Metadata only best threshold: 0.5008835545309599


,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_default_0_5,0.5000,0.4060,0.3909,0.3845,0.4301,0.5358,529,325,269,203
1,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_tuned_min_0_5,0.5067,0.4068,0.3909,0.3877,0.4280,0.5358,535,319,270,202
2,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_default_0_5,0.5000,0.4048,0.4307,0.4146,0.3953,0.5810,590,264,286,187
3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_tuned_min_0_5,0.5067,0.3974,0.4307,0.4108,0.3848,0.5810,593,261,291,182
4,ablation_mlp_metadata_only,metadata_only,validation_default_0_5,0.5000,0.4248,0.3809,0.3918,0.4640,0.5533,514,340,253,219
5,ablation_mlp_metadata_only,metadata_only,validation_tuned_min_0_5,0.5009,0.4257,0.3809,0.3932,0.4640,0.5533,516,338,253,219
6,ablation_mlp_metadata_only,metadata_only,test_default_0_5,0.5000,0.4621,0.4069,0.4144,0.5222,0.5558,505,349,226,247
7,ablation_mlp_metadata_only,metadata_only,test_tuned_min_0_5,0.5009,0.4615,0.4069,0.4148,0.5201,0.5558,507,347,227,246


## 7. 메타데이터 영향도 분석

04번에서 저장한 제안 모델을 불러와 validation 데이터 기준 permutation importance를 계산한다.

메타데이터 컬럼을 하나씩 섞었을 때 F1이 얼마나 떨어지는지 확인한다. F1 감소폭이 클수록 해당 메타데이터가 모델 예측에 더 큰 영향을 준 것으로 해석한다.

In [ ]:
def metadata_permutation_importance(model, X_val, y_val, metadata_cols, threshold, repeats=10, random_state=SEED):
    rng = np.random.default_rng(random_state)
    base_prob = model.predict_proba(X_val)[:, 1]
    base_pred = (base_prob >= threshold).astype(int)
    base_f1 = f1_score(y_val, base_pred)

    rows = []
    for col in metadata_cols:
        drops = []
        for _ in range(repeats):
            X_perm = X_val.copy()
            shuffled = X_perm[col].to_numpy().copy()
            rng.shuffle(shuffled)
            X_perm[col] = shuffled

            perm_prob = model.predict_proba(X_perm)[:, 1]
            perm_pred = (perm_prob >= threshold).astype(int)
            perm_f1 = f1_score(y_val, perm_pred)
            drops.append(base_f1 - perm_f1)

        rows.append({
            'metadata': col,
            'base_f1': float(base_f1),
            'mean_f1_drop': float(np.mean(drops)),
            'std_f1_drop': float(np.std(drops)),
            'repeats': repeats,
        })

    return pd.DataFrame(rows).sort_values('mean_f1_drop', ascending=False).reset_index(drop=True)


proposed_bundle = joblib.load('outputs/proposed_mlp_final_model.joblib')
proposed_model = proposed_bundle['model']
proposed_feature_cols = proposed_bundle['feature_cols']
proposed_threshold = proposed_bundle.get('best_threshold', proposed_config['best_threshold_from_validation'])

metadata_importance = metadata_permutation_importance(
    proposed_model,
    X_val[proposed_feature_cols],
    y_val,
    meta_cols,
    proposed_threshold,
    repeats=10,
)

metadata_importance_display = metadata_importance.copy()
metadata_importance_display.insert(0, 'rank', metadata_importance_display.index + 1)
metadata_importance_display = metadata_importance_display.rename(columns={
    'metadata': '메타데이터',
    'base_f1': '기준 F1',
    'mean_f1_drop': '평균 F1 감소폭',
    'std_f1_drop': 'F1 감소폭 표준편차',
    'repeats': '반복 횟수',
})
metadata_importance_display = metadata_importance_display.round({
    '기준 F1': 4,
    '평균 F1 감소폭': 4,
    'F1 감소폭 표준편차': 4,
})

top_metadata = metadata_importance.iloc[0]
print(
    f"가장 큰 영향을 준 메타데이터: {top_metadata['metadata']} "
    f"(평균 F1 감소폭: {top_metadata['mean_f1_drop']:.4f})"
)
display(metadata_importance_display)

Most influential metadata: photo_count (mean F1 drop: 0.0180)


,rank,metadata,base_f1,mean_f1_drop,std_f1_drop,repeats
0,1,photo_count,0.4444,0.0180,0.0073,10
1,2,text_length,0.4444,0.0080,0.0045,10
2,3,emoji_count,0.4444,0.0038,0.0035,10


## 8. 전체 비교표 출력 및 모델 저장

04번 제안 모델 결과와 05번에서 학습한 baseline/ablation 결과를 하나의 표로 합친다.

모델 선택용 순위는 `validation_tuned_min_0_5` 기준을 우선 확인한다. `test_tuned_min_0_5`는 최종 선택 이후 일반화 성능을 확인하기 위한 참고 표로만 본다.

성능 비교 결과는 노트북 셀에 바로 출력하고, 모델 객체만 .joblib 파일로 따로 저장한다.

In [ ]:
baseline_metrics = pd.concat([
    proposed_metrics,
    pd.DataFrame(tfidf_rows),
    pd.DataFrame(pca_only_rows),
    pd.DataFrame(metadata_only_rows),
], ignore_index=True)

validation_summary = (
    baseline_metrics[baseline_metrics['split'] == 'validation_tuned_min_0_5']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
validation_summary.insert(0, 'rank', validation_summary.index + 1)

test_summary = (
    baseline_metrics[baseline_metrics['split'] == 'test_tuned_min_0_5']
    .sort_values(['f1', 'pr_auc'], ascending=False)
    .reset_index(drop=True)
)
test_summary.insert(0, 'rank', test_summary.index + 1)

validation_summary_display = validation_summary.round({
    'threshold': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
    'roc_auc': 4,
})

test_summary_display = test_summary.round({
    'threshold': 4,
    'f1': 4,
    'pr_auc': 4,
    'precision': 4,
    'recall': 4,
    'roc_auc': 4,
})

baseline_metrics_display = (
    baseline_metrics
    .sort_values(['model', 'split'])
    .reset_index(drop=True)
    .round({
        'threshold': 4,
        'f1': 4,
        'pr_auc': 4,
        'precision': 4,
        'recall': 4,
        'roc_auc': 4,
    })
)

baseline_metrics.to_csv('outputs/baseline_comparison_metrics.csv', index=False, encoding='utf-8-sig')
validation_summary.to_csv('outputs/baseline_validation_selection_summary.csv', index=False, encoding='utf-8-sig')
test_summary.to_csv('outputs/baseline_test_reference_summary.csv', index=False, encoding='utf-8-sig')

saved_model_files = pd.DataFrame([
    {'model': 'baseline_tfidf_random_forest', 'path': 'outputs/baseline_tfidf_random_forest_model.joblib'},
    {'model': 'ablation_mlp_kcbert_pca_only', 'path': 'outputs/ablation_mlp_kcbert_pca_only_model.joblib'},
    {'model': 'ablation_mlp_metadata_only', 'path': 'outputs/ablation_mlp_metadata_only_model.joblib'},
    {'model': 'proposed_hybrid_mlp_04', 'path': 'outputs/proposed_mlp_final_model.joblib'},
])

print('Validation tuned 기준 모델 선택용 순위')
display(validation_summary_display)

print('Test tuned 기준 참고 순위')
display(test_summary_display)

print('Validation/Test 전체 평가 결과')
display(baseline_metrics_display)

print('저장된 모델 파일')
display(saved_model_files)

Validation tuned ranking for model selection


,rank,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,proposed_hybrid_mlp_04,kcbert_pca+metadata,validation_tuned_min_0_5,0.5038,0.4444,0.4070,0.4116,0.4831,0.5561,528,326,244,228
1,2,ablation_mlp_metadata_only,metadata_only,validation_tuned_min_0_5,0.5009,0.4257,0.3809,0.3932,0.4640,0.5533,516,338,253,219
2,3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_tuned_min_0_5,0.5067,0.4068,0.3909,0.3877,0.4280,0.5358,535,319,270,202
3,4,baseline_tfidf_random_forest,cleaned_text_tfidf,validation_tuned_min_0_5,0.5016,0.2613,0.4015,0.4485,0.1843,0.5482,747,107,385,87


Test tuned ranking for reference


,rank,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,1,ablation_mlp_metadata_only,metadata_only,test_tuned_min_0_5,0.5009,0.4615,0.4069,0.4148,0.5201,0.5558,507,347,227,246
1,2,proposed_hybrid_mlp_04,kcbert_pca+metadata,test_tuned_min_0_5,0.5038,0.4271,0.4423,0.4152,0.4397,0.5784,561,293,265,208
2,3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_tuned_min_0_5,0.5067,0.3974,0.4307,0.4108,0.3848,0.5810,593,261,291,182
3,4,baseline_tfidf_random_forest,cleaned_text_tfidf,test_tuned_min_0_5,0.5016,0.2927,0.4381,0.5246,0.2030,0.5837,767,87,377,96


All validation/test metrics


,model,feature_set,split,threshold,f1,pr_auc,precision,recall,roc_auc,tn,fp,fn,tp
0,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_default_0_5,0.5000,0.4048,0.4307,0.4146,0.3953,0.5810,590,264,286,187
1,ablation_mlp_kcbert_pca_only,kcbert_pca_only,test_tuned_min_0_5,0.5067,0.3974,0.4307,0.4108,0.3848,0.5810,593,261,291,182
2,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_default_0_5,0.5000,0.4060,0.3909,0.3845,0.4301,0.5358,529,325,269,203
3,ablation_mlp_kcbert_pca_only,kcbert_pca_only,validation_tuned_min_0_5,0.5067,0.4068,0.3909,0.3877,0.4280,0.5358,535,319,270,202
4,ablation_mlp_metadata_only,metadata_only,test_default_0_5,0.5000,0.4621,0.4069,0.4144,0.5222,0.5558,505,349,226,247
5,ablation_mlp_metadata_only,metadata_only,test_tuned_min_0_5,0.5009,0.4615,0.4069,0.4148,0.5201,0.5558,507,347,227,246
6,ablation_mlp_metadata_only,metadata_only,validation_default_0_5,0.5000,0.4248,0.3809,0.3918,0.4640,0.5533,514,340,253,219
7,ablation_mlp_metadata_only,metadata_only,validation_tuned_min_0_5,0.5009,0.4257,0.3809,0.3932,0.4640,0.5533,516,338,253,219
8,baseline_tfidf_random_forest,cleaned_text_tfidf,test_default_0_5,0.5000,0.2918,0.4381,0.5189,0.2030,0.5837,765,89,377,96
9,baseline_tfidf_random_forest,cleaned_text_tfidf,test_tuned_min_0_5,0.5016,0.2927,0.4381,0.5246,0.2030,0.5837,767,87,377,96


Saved model files


,model,path
0,baseline_tfidf_random_forest,outputs/baseline_tfidf_random_forest_model.joblib
1,ablation_mlp_kcbert_pca_only,outputs/ablation_mlp_kcbert_pca_only_model.joblib
2,ablation_mlp_metadata_only,outputs/ablation_mlp_metadata_only_model.joblib
3,proposed_hybrid_mlp_04,outputs/proposed_mlp_final_model.joblib


## 9. 05번 결과 해석 기준

05번 실행 후에는 다음을 확인한다.

1. validation 기준에서 제안 모델이 `baseline_tfidf_random_forest`보다 좋은가?
2. validation 기준에서 제안 모델이 `ablation_mlp_kcbert_pca_only`보다 좋은가?
3. validation 기준에서 제안 모델이 `ablation_mlp_metadata_only`보다 좋은가?
4. metadata를 결합했을 때 F1, PR-AUC, Recall이 실제로 올라갔는가?
5. 메타데이터 영향도 출력표에서 어떤 메타데이터가 예측에 가장 큰 영향을 줬는가?

이 결과가 06번의 입력이 된다.

06번에서는 다음 작업을 진행한다.

- validation 기준 최종 모델 선택
- 선택 모델의 test 성능 최종 확인
- FP/FN 오답 리뷰 추출
- 은어, 신조어, 우회 표현에 대한 실패 사례 분석
- 메타데이터가 텍스트 기반 모호성을 줄였는지 해석

따라서 05번은 별점 정제 단계가 아니라, 제안 모델의 타당성을 검증하는 단계이다.